## Support Vector Machine

Finds the **maximum margin hyperplane**, where "margin" refers to the distance between the hyperplane and the closest data points (called **support vectors**).

<br>

<p align="center">
<img src="visualizations/SVM.png" width="600">
</p>

---

### 1. **Linear SVM (linearly separable data)**

Given labeled data:

* Features: $\mathbf{x}_i \in \mathbb{R}^n$
* Labels: $y_i \in \{-1, +1\}$

SVM tries to find a hyperplane defined by:

$$
\mathbf{w}^\top \mathbf{x} + b = 0
$$

That separates the two classes while maximizing the margin $\frac{2}{||\mathbf{w}||}$.

**Optimization Problem**:

$$
\min_{\mathbf{w}, b} \frac{1}{2} ||\mathbf{w}||^2
$$

Subject to:

$$
y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1
$$

The data points that lie on the margin boundary (where $y\_i(\mathbf{w}^\top \mathbf{x}\_i + b) = 1$) are called **support vectors**.

---

### 2. **Non-linearly Separable Data & Soft Margin**

When data isn’t perfectly separable, SVM uses a **soft margin** that allows some misclassifications:

$$
\min_{\mathbf{w}, b, \xi} \frac{1}{2} ||\mathbf{w}||^2 + C \sum_{i=1}^{n} \xi_i
$$

Subject to:

$$
y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \xi_i,\quad \xi_i \geq 0
$$

Where:

* $\xi_i$: slack variables that allow violations.
* $C$: penalty parameter that controls trade-off between margin size and misclassification.
---

### 3. **Multiclass Classification**

Standard SVMs are binary classifiers. To extend them to multiclass problems, several strategies can be used:

#### a) One-vs-All (OvA):

Train $K$ binary classifiers (for $K$ classes). For class $k$:

$$
y_i^{(k)} = 
\begin{cases}
+1 & \text{if } y_i = k \\
-1 & \text{otherwise}
\end{cases}
$$

The class with the highest score $\mathbf{w}_k^\top \mathbf{x} + b_k$ is predicted.

#### b) One-vs-One (OvO):

Train $\frac{K(K-1)}{2}$ classifiers for all class pairs. Prediction is made by majority vote among classifiers.

#### c) Structured or Vectorized Multiclass SVM:

Directly define a joint optimization problem using class scores:

Let:

* $\mathbf{W} \in \mathbb{R}^{K \times d}$: weight matrix, where each row corresponds to a class.
* For sample $\mathbf{x}_i$, compute scores: $s = \mathbf{W} \mathbf{x}_i \in \mathbb{R}^K$

Use the **multiclass hinge loss**:

$$
L_i = \sum_{j \ne y_i} \max(0, s_j - s_{y_i} + 1)
$$

Final loss (average + regularization):

$$
\mathcal{L} = \frac{1}{N} \sum_{i=1}^N L_i + \frac{\lambda}{2} ||\mathbf{W}||^2
$$

This approach is commonly used in practical implementations for multiclass SVMs.

---

## ✅ Advantages

* Effective in high-dimensional spaces.
* Works well when there’s a clear margin of separation.
* Memory efficient (only support vectors matter).
* Can model non-linear decision boundaries with kernels.

---

## ⚠️ Disadvantages

* Doesn’t scale well with large datasets (training time complexity is high).
* Choosing the right parameters can be tricky.
* Not easily interpretable compared to decision trees or logistic regression.

In [41]:
import numpy as np
from tqdm import tqdm
from cifar10.unpickle import get_all_data, get_test_data

In [42]:
# 1) Load
x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

y_train = np.array(y_train)
y_test = np.array(y_test)

# 2) Normalize to [0,1] and flatten
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

n_samples, h, w, c = x_train.shape  # h=32, w=32, c=3
x_train = x_train.reshape(n_samples, h * w * c)  # flatten to (N, 3072)
x_test = x_test.reshape(x_test.shape[0], h * w * c)  # flatten to (N, 3072)

In [ ]:
class LinearSVM:
    def __init__(self, input_dim, num_classes, lr=1e-3, reg=1e-4):
        """
        input_dim: data dimensionality (without bias)
        num_classes: number of classes
        lr: learning rate
        reg: L2 regularization strength
        """
        # include bias term
        self.input_dim = input_dim + 1
        self.W = 0.001 * np.random.randn(num_classes, self.input_dim)  # weight matrix
        self.lr = lr
        self.reg = reg

    def compute_loss_and_gradients(self, X, y):
        """
        X: (batch_size, D) with bias column appended
        y: (batch_size,) labels
        Returns loss and gradient dW of shape (num_classes, D)
        """
        num_train = X.shape[0]
        scores = self.W.dot(X.T)  # (num_classes, batch_size)
        correct_scores = scores[y, np.arange(num_train)]

        # compute margins for all classes
        margins = np.maximum(0, scores - correct_scores + 1.0)
        margins[y, np.arange(num_train)] = 0  # do not count correct class
        data_loss = np.sum(margins) / num_train
        reg_loss = 0.5 * self.reg * np.sum(self.W * self.W)
        loss = data_loss + reg_loss

        # gradient calculation using binary mask
        binary = (margins > 0).astype(float)
        row_sum = np.sum(binary, axis=0)
        binary[y, np.arange(num_train)] = -row_sum
        dW = binary.dot(X) / num_train
        dW += self.reg * self.W  # regularization gradient

        return loss, dW

    def train(self, X, y, epochs=15, batch_size=256, verbose=True):
        """
        X: (N, original_input_dim) WITHOUT bias
        y: (N,) labels
        """
        # append bias column
        X_bias = np.hstack([X, np.ones((X.shape[0], 1))])
        N = X_bias.shape[0]

        for epoch in range(epochs):
            perm = np.random.permutation(N)
            X_shuffled = X_bias[perm]
            y_shuffled = y[perm]

            running_loss = 0.0
            num_batches = 0

            for i in range(0, N, batch_size):
                X_batch = X_shuffled[i : i + batch_size]
                y_batch = y_shuffled[i : i + batch_size]

                loss, grad = self.compute_loss_and_gradients(X_batch, y_batch)
                self.W -= self.lr * grad  # SGD update

                running_loss += loss
                num_batches += 1

            avg_loss = running_loss / num_batches
            if verbose:
                print(f"Epoch {epoch + 1}/{epochs}, avg_loss: {avg_loss:.4f}")

    def predict(self, X):
        """
        X: (num_samples, original_input_dim) WITHOUT bias
        Returns predicted labels array of shape (num_samples,)
        """
        # append bias column
        X_bias = np.hstack([X, np.ones((X.shape[0], 1))])
        scores = self.W.dot(X_bias.T)
        y_pred = np.argmax(scores, axis=0)
        return y_pred

In [44]:
svm = LinearSVM(input_dim=3072, num_classes=10, lr=1e-3, reg=1e-4)
svm.train(x_train, y_train, epochs=10, batch_size=200)

Epoch 1/10, avg_loss: 6.1394
Epoch 2/10, avg_loss: 5.2208
Epoch 3/10, avg_loss: 4.9931
Epoch 4/10, avg_loss: 4.8682
Epoch 5/10, avg_loss: 4.7868
Epoch 6/10, avg_loss: 4.7318
Epoch 7/10, avg_loss: 4.6794
Epoch 8/10, avg_loss: 4.6422
Epoch 9/10, avg_loss: 4.6100
Epoch 10/10, avg_loss: 4.5758


In [45]:
y_pred = svm.predict(x_test)
accuracy = np.mean(y_pred == y_test)
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.3606


<h1 style="font-size: 40px;">Integrate filters</h1>

In [46]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from images.image_preprocessing import (
    extract_raw_pixels,
    extract_color_histogram,
    extract_hog,
    extract_lbp,
)

In [47]:
def batch_extract(fn, X):
    """Apply single‐image fn over a batch X of shape (n,H,W,C)."""
    return np.stack([fn(im) for im in X], axis=0)


x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

Xraw_train = batch_extract(extract_raw_pixels, x_train)
Xraw_test = batch_extract(extract_raw_pixels, x_test)

Xhist_train = batch_extract(extract_color_histogram, x_train)
Xhist_test = batch_extract(extract_color_histogram, x_test)

Xhog_train = batch_extract(extract_hog, x_train)
Xhog_test = batch_extract(extract_hog, x_test)

Xlbp_train = batch_extract(extract_lbp, x_train)
Xlbp_test = batch_extract(extract_lbp, x_test)

In [48]:
X_train = np.concatenate([Xraw_train, Xhist_train, Xhog_train, Xlbp_train], axis=1)
X_test = np.concatenate([Xraw_test, Xhist_test, Xhog_test, Xlbp_test], axis=1)

mean_all = np.mean(X_train, axis=0)
std_all = np.std(X_train, axis=0)

eps = 1e-8


X_train = (X_train - mean_all) / (std_all + eps)
X_test = (X_test - mean_all) / (std_all + eps)


y_train = np.array(y_train)
y_test = np.array(y_test)

In [51]:
svm = LinearSVM(input_dim=7502, num_classes=10, lr=1e-4, reg=1e-3)
svm.train(X_train, y_train, epochs=15, batch_size=200)

Epoch 1/15, avg_loss: 4.7674
Epoch 2/15, avg_loss: 3.4613
Epoch 3/15, avg_loss: 3.0506
Epoch 4/15, avg_loss: 2.8160
Epoch 5/15, avg_loss: 2.6587
Epoch 6/15, avg_loss: 2.5431
Epoch 7/15, avg_loss: 2.4538
Epoch 8/15, avg_loss: 2.3824
Epoch 9/15, avg_loss: 2.3220
Epoch 10/15, avg_loss: 2.2711
Epoch 11/15, avg_loss: 2.2276
Epoch 12/15, avg_loss: 2.1884
Epoch 13/15, avg_loss: 2.1547
Epoch 14/15, avg_loss: 2.1227
Epoch 15/15, avg_loss: 2.0957


In [52]:
y_pred = svm.predict(X_test)
accuracy = np.mean(y_pred == y_test)
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.6026
